In [1]:
# 1. 필요한 라이브러리를 불러온다.
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import (
    get_peft_model, 
    LoraConfig, 
    TaskType, 
    prepare_model_for_kbit_training
)

d:\dev\workspace\ai\llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
torch.cuda.is_available()

False

In [3]:
df = pd.read_csv("./data/review_data.csv", encoding="cp949")

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["labels"],
    random_state=0
)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [5]:
# 4. 기본적인 모델명과 토크나이저를 설정하고, QLoRA 전용 설정을 한다.
model_id = "beomi/kcbert-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          
    bnb_4bit_use_double_quant=True,  
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=2,
    #quantization_config=bnb_config,
    torch_dtype=torch.float32,
    device_map="auto"
)

base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
base_model.config.use_cache = False

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 861.11it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

In [6]:
# 5. 불러온 데이터를 토크나이저를 활용해 전처리한다.
def preprocess(data):
    return tokenizer(
        data["text"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

train_dataset = train_dataset.map(
    preprocess,
    batched=True
    #remove_columns=["text", "__index_level_0__"]
)

test_dataset = test_dataset.map(
    preprocess,
    batched=True
    #remove_columns=["text", "__index_level_0__"]
)

Map: 100%|██████████| 20/20 [00:00<00:00, 275.41 examples/s]


In [7]:
# 6. 앞서 양자화해서 불러온 base_model에 LoRA를 적용해 최종적으로 QLoRA 모델을 생성한다.
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    target_modules=["query", "value"]
)

qlora_model = get_peft_model(base_model, peft_config)
qlora_model.print_trainable_parameters()

trainable params: 296,450 || all params: 109,216,516 || trainable%: 0.2714


In [8]:
# 7. 학습하기 전 학습에 필요한 설정을 한다.
training_args = TrainingArguments(
    output_dir="./saved_models/qlora_sentiment",

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,

    logging_strategy="epoch",

    fp16=True,
    report_to="none"
)

def compute_metrics(predict):
    preds = np.argmax(predict.predictions, axis=1)
    acc = np.mean(preds == predict.label_ids)
    return {"accuracy": acc}

trainer = Trainer(
    model=qlora_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
# 8. 학습을 진행한다.
trainer.train()

d:\dev\workspace\ai\llm\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.716026,0.700267,0.500000
2,0.727071,0.689295,0.500000
3,0.691506,0.683306,0.550000
4,0.679395,0.678015,0.550000
5,0.677886,0.676596,0.550000


d:\dev\workspace\ai\llm\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
d:\dev\workspace\ai\llm\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
d:\dev\workspace\ai\llm\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
d:\dev\workspace\ai\llm\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=50, training_loss=0.6983768653869629, metrics={'train_runtime': 554.3005, 'train_samples_per_second': 0.722, 'train_steps_per_second': 0.09, 'total_flos': 13201087488000.0, 'train_loss': 0.6983768653869629, 'epoch': 5.0})

In [24]:
# 테스트 예측
test_texts = [
    "전체적인 분위기가 좋아서 편하게 볼 수 있었어요.",
    "스토리는 평범했지만 연출 덕분에 재미있었어요.",
    "배우들의 연기가 자연스러워서 몰입이 잘 됐어요.",
    "큰 기대 없이 봤는데 생각보다 괜찮았어요.",
    "잔잔하지만 끝나고 나서 여운이 남는 영화였어요.",

    "이야기가 늘어져서 중간부터 집중이 안 됐어요.",
    "연출이 과해서 오히려 몰입을 방해했어요.",
    "캐릭터 행동이 이해되지 않아서 답답했어요.",
    "분위기는 잡으려는 것 같은데 내용이 부족했어요.",
    "전체적으로 뭔가 아쉬운 느낌이 많이 남았어요."
]

inputs = tokenizer(
    test_texts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=64
).to(qlora_model.device)

In [25]:
qlora_model.eval()
with torch.no_grad():
    outputs = qlora_model(**inputs)

preds = torch.argmax(outputs.logits, dim=1)
print("예측 결과:", preds.tolist())

예측 결과: [0, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [26]:
labels_target = torch.tensor([1, 1, 1, 1, 1, 0, 0, 0, 0, 0]).to(preds.device)
accuracy = (preds == labels_target).float().mean()
print("Accuracy:", accuracy.item())

Accuracy: 0.4000000059604645


In [27]:
trainer.save_model("./saved_models/qlora_sentiment/final_model")
tokenizer.save_pretrained("./saved_models/qlora_sentiment/final_model")

('./saved_models/qlora_sentiment/final_model\\tokenizer_config.json',
 './saved_models/qlora_sentiment/final_model\\tokenizer.json')